In [ ]:
# hide
# --- Dependencies -----------------------------------------------------------
# Everything this notebook imports, installed up front. The installs are
# no-ops when the packages are already present (they ship with the book's
# build environment and with the browser kernel), so this cell just needs to
# run once. The capture_output() block hides pip's chatter but still lets a
# genuine install failure raise. myst-nb isn't installed here — it's build-time
# only, and the browser kernel stubs its glue() to a no-op.
from IPython.utils.capture import capture_output
with capture_output():                 # swallow pip + import chatter (e.g. matplotlib's font cache); real failures still raise
    # numpy + matplotlib are Pyodide built-ins; the install is a no-op at build time
    %pip install -q numpy matplotlib
    import numpy as np
    import matplotlib.pyplot as plt
    import pyquist as pq
    from matplotlib.animation import FuncAnimation
    from myst_nb import glue
    from IPython.display import HTML

In [ ]:
# collapse
# --- House plotting style ----------------------------------------------------
# Every widget starts from this rcParams block: dpi 64 keeps the embedded
# jshtml player a sane size, mathtext avoids a LaTeX requirement, and the
# open spines match the book's figures.
plt.rcParams.update({
    "figure.dpi": 64,                # sets the player's intrinsic size only —
    "figure.subplot.bottom": 0.18,   # reserve room for x-axis labels (jshtml crops at the figure bbox)
    "animation.html": "jshtml",      # SVG frames are vector, sharp at any zoom
    "animation.frame_format": "svg", # ~3x smaller gzipped than PNG, and crisp
    "animation.embed_limit": 40,     # MB; matplotlib's 20 MB default drops frames
    "text.usetex": False,            # mathtext: no LaTeX install needed locally
    "font.size": 12,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#6D6E71",
    "xtick.color": "#6D6E71", "ytick.color": "#6D6E71",
    "xtick.labelcolor": "#3b3b3b", "ytick.labelcolor": "#3b3b3b",
    "axes.labelcolor": "#3b3b3b",
})

# The book's plot palette (CMU's colors) — see the style guide on this page.
RED, BLUE, GOLD, IRON, TEAL, STEEL = "#C41230", "#007BC0", "#FDB515", "#6D6E71", "#008F91", "#E0E0E0"

Each frame adds the next harmonic of the sawtooth recipe — harmonic $k$
at amplitude $1/k$ — to a running sum. On the left, the partial sum
(Carnegie red) bends closer to the ideal sawtooth (steel gray), with the
newest harmonic drawn in gold; on the right, each bar of the recipe
lights up as its harmonic joins. Press **play**.

In [ ]:
# --- The widget: a sawtooth assembling itself, harmonic by harmonic ----------
f0 = 220.0                            # fundamental (A3)
n_harmonics = 12
T = 2 / f0                            # two cycles on screen
t = np.linspace(0.0, T, 900, endpoint=False)

# The sawtooth recipe: harmonic k at amplitude 1/k (2/pi normalizes the full
# series to peak +-1). Row n of `sums` is the wave after harmonics 1..n+1.
k = np.arange(1, n_harmonics + 1)
partials = (2 / np.pi) * np.sin(2 * np.pi * f0 * k[:, None] * t[None, :]) / k[:, None]
sums = np.cumsum(partials, axis=0)
ideal = 1.0 - 2.0 * ((f0 * t) % 1.0)  # what the infinite series converges to

fig, (ax_w, ax_s) = plt.subplots(
    1, 2, figsize=(8.2, 3.4), gridspec_kw={"width_ratios": [1.6, 1], "wspace": 0.3},
)

# left: the ideal sawtooth as a fixed reference, the partial sum drawn on top
ax_w.plot(t * 1000, ideal, color=STEEL, lw=1.6)
wave, = ax_w.plot(t * 1000, sums[0], color=RED, lw=2.0)
newest, = ax_w.plot(t * 1000, partials[0], color=GOLD, lw=1.4, alpha=0.9)
ax_w.set_xlim(0, T * 1000); ax_w.set_ylim(-1.35, 1.35)
ax_w.set_xlabel("time (ms)"); ax_w.set_ylabel("amplitude")
title = ax_w.set_title("1 harmonic")

# right: the recipe — each bar lights up as its harmonic joins the sum
bars = ax_s.bar(k, 1 / k, width=0.6, color=STEEL)
ax_s.set_xticks([1, 4, 8, 12])
ax_s.set_xlabel("harmonic $k$"); ax_s.set_ylabel("amplitude $1/k$")
ax_s.set_ylim(0, 1.05)


def animate(n):
    wave.set_ydata(sums[n])
    newest.set_ydata(partials[n])
    title.set_text(f"{n + 1} harmonic" + ("s" if n else ""))
    for j, bar in enumerate(bars):
        bar.set_color(GOLD if j == n else RED if j < n else STEEL)
    return ()


frame_rate = 2                        # one new harmonic every half second
anim = FuncAnimation(fig, animate, frames=n_harmonics,
                     interval=1000 / frame_rate, blit=False)
player = HTML(anim.to_jshtml())
plt.close(fig)                        # only the player reaches the page

# Register the player for re-embedding elsewhere; the trailing expression
# bakes it into the page.
glue("harmonics-builder", player, display=False)
player

**Now hear it.** The same recipe rendered as sound: half a second each
of 1, 2, 4, 8, then 16 harmonics of the same fundamental. Listen for the
tone sharpening toward a buzz as the upper harmonics arrive.

In [ ]:
# --- The same recipe, as sound ------------------------------------------------
sr = 44100
seg = np.arange(int(0.5 * sr)) / sr   # half a second per step
steps = [1, 2, 4, 8, 16]

clips = []
for n in steps:
    kk = np.arange(1, n + 1)
    saw = (np.sin(2 * np.pi * f0 * kk[:, None] * seg) / kk[:, None]).sum(axis=0)
    saw *= 0.5 / np.abs(saw).max()    # consistent, comfortable loudness
    saw[:220] *= np.linspace(0, 1, 220)   # 5 ms fades: no clicks between steps
    saw[-220:] *= np.linspace(1, 0, 220)
    clips.append(saw)

pq.play(pq.Audio(np.concatenate(clips), sample_rate=sr))